# Задание 2. Classification with Sparse Features

В этом ноутбуке реализована **вся Task 2 одним файлом**:

1. загрузка Book Recommendation Dataset;
2. построение sparse user-book interaction matrix;
3. разбиение на train / validation / test;
4. подбор и обучение Ridge;
5. подбор и обучение Random Forest;
6. линейное снижение размерности для sparse-данных;
7. UMAP;
8. повторное обучение Ridge и Random Forest на сжатых признаках;
9. сравнение качества и времени обучения.

По условию проекта `Ratings.csv` используется для формирования user-book features, а `Users.csv` — для получения возраста пользователя как target.


## Важное техническое решение: sparse PCA

Исходная матрица имеет сотни тысяч признаков и является разреженной. Классический `sklearn.decomposition.PCA` требует плотное представление данных, а преобразование такой матрицы в dense может потребовать десятки гигабайт памяти.

Поэтому для основной экспериментальной линии используется **`TruncatedSVD`**, который работает непосредственно со sparse CSR matrix и выполняет линейное снижение размерности. Для центрированных данных PCA может вычисляться через SVD, поэтому здесь `TruncatedSVD` используется как вычислительно безопасный sparse-аналог PCA.

Это решение явно фиксируем в ноутбуке, чтобы на ревью было понятно, почему мы не делаем `PCA.fit(X)` на матрице размерности порядка `340556`.


In [1]:
%pip install umap-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz, save_npz

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

try:
    import umap.umap_ as umap
except ImportError as exc:
    raise ImportError(
        "Не найден пакет umap-learn. Установи его командой: pip install umap-learn"
    ) from exc

RANDOM_STATE = 42

# Ограничение количества пользователей для тяжёлых экспериментов.
# Полная sparse matrix всё равно строится на всех подходящих пользователях.
# После проверки pipeline значение можно увеличить или поставить None.
MODEL_SAMPLE_SIZE = 12000

# Размерность линейного sparse-снижения.
SVD_COMPONENTS = 50

# Размерность UMAP-представления для последующего ML.
UMAP_COMPONENTS = 20

# Количество соседей UMAP.
UMAP_NEIGHBORS = 15

# RF deliberately kept moderate because 340k sparse features are expensive.
RF_N_ESTIMATORS = 100

np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", None)


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Автоматически находим data независимо от рабочей директории Jupyter

DATA_CANDIDATES = [
    Path("data"),
    Path("../data"),
    Path("../../data"),
]

DATA_DIR = next(
    (
        path
        for path in DATA_CANDIDATES
        if (path / "Book_Recommendation" / "Ratings.csv").exists()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Не найдена папка data/Book_Recommendation. "
        "Проверь структуру проекта."
    )

BOOK_DIR = DATA_DIR / "Book_Recommendation"
RATINGS_PATH = BOOK_DIR / "Ratings.csv"
USERS_PATH = BOOK_DIR / "Users.csv"

print("DATA_DIR:", DATA_DIR.resolve())
print("Ratings:", RATINGS_PATH.resolve())
print("Users:", USERS_PATH.resolve())


DATA_DIR: C:\Users\User\Desktop\ML6_Unsupervised_learning_ID_1254805-1\src\data
Ratings: C:\Users\User\Desktop\ML6_Unsupervised_learning_ID_1254805-1\src\data\Book_Recommendation\Ratings.csv
Users: C:\Users\User\Desktop\ML6_Unsupervised_learning_ID_1254805-1\src\data\Book_Recommendation\Users.csv


## 1. Загрузка исходных данных

In [4]:
ratings = pd.read_csv(
    RATINGS_PATH,
    dtype={
        "User-ID": "int32",
        "ISBN": "string",
        "Book-Rating": "int8",
    },
)

users = pd.read_csv(
    USERS_PATH,
    dtype={
        "User-ID": "int32",
        "Age": "float32",
    },
)

print("Ratings shape:", ratings.shape)
print("Users shape:", users.shape)

display(ratings.head())
display(users.head())


Ratings shape: (1149780, 3)
Users shape: (278858, 3)


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


## 2. Первичный анализ

Проверим размер данных, количество уникальных пользователей и книг, значения рейтинга и пропуски в `Age`.


In [5]:
print("Уникальных пользователей в Ratings:", ratings["User-ID"].nunique())
print("Уникальных ISBN в Ratings:", ratings["ISBN"].nunique())
print("Количество строк Ratings:", len(ratings))
print("Дубликатов полных строк:", ratings.duplicated().sum())

print("\nРаспределение Book-Rating:")
display(ratings["Book-Rating"].value_counts().sort_index())

print("\nВозраст:")
print("Пропусков Age:", users["Age"].isna().sum())
display(users["Age"].describe())


Уникальных пользователей в Ratings: 105283
Уникальных ISBN в Ratings: 340556
Количество строк Ratings: 1149780
Дубликатов полных строк: 0

Распределение Book-Rating:


Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64


Возраст:
Пропусков Age: 110762


count    168096.000000
mean         34.751434
std          14.428098
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64

### Контрольная точка чек-листа

В исходном `Ratings.csv` должно быть **340 556 уникальных ISBN**. Именно столько признаков-книг ожидается в чек-листе проекта.

В наших экспериментах эта размерность сохраняется: один столбец матрицы `X` соответствует одной книге.


In [6]:
N_BOOKS_EXPECTED = 340_556
n_unique_books = ratings["ISBN"].nunique()

print(f"Уникальных книг: {n_unique_books:,}")
assert n_unique_books == N_BOOKS_EXPECTED, (
    f"Ожидалось {N_BOOKS_EXPECTED}, получено {n_unique_books}"
)


Уникальных книг: 340,556


## 3. Подготовка target `Age`

В `Users.csv` встречаются пропуски и явно аномальные значения возраста. Для обучения используем диапазон `5..100` лет.

Это решение влияет только на количество строк (`пользователей`), но **не уменьшает число признаков-книг**.


In [7]:
VALID_AGE_MIN = 5
VALID_AGE_MAX = 100

users_valid = users.loc[
    users["Age"].between(VALID_AGE_MIN, VALID_AGE_MAX),
    ["User-ID", "Age"],
].copy()

ratings_user_ids = ratings["User-ID"].unique()
users_valid = users_valid[
    users_valid["User-ID"].isin(ratings_user_ids)
].copy()

print("Пользователей с корректным Age и взаимодействиями:", len(users_valid))
print("Уникальных User-ID:", users_valid["User-ID"].nunique())
display(users_valid["Age"].describe())


Пользователей с корректным Age и взаимодействиями: 61640
Уникальных User-ID: 61640


count    61640.000000
mean        35.445248
std         13.769306
min          5.000000
25%         25.000000
50%         33.000000
75%         45.000000
max        100.000000
Name: Age, dtype: float64

## 4. Построение sparse user-book interaction matrix

Признак определяется как **факт взаимодействия пользователя с книгой**.

Поэтому:

- есть запись `(User-ID, ISBN)` → `1`;
- нет записи → `0`.

Само числовое значение `Book-Rating` не используется как величина признака.

Mapping ISBN создаём по **всему** `Ratings.csv`, чтобы сохранить все 340 556 признаков.


In [8]:
# Mapping пользователей
user_ids = users_valid["User-ID"].to_numpy()
user_to_row = {user_id: row for row, user_id in enumerate(user_ids)}

# Mapping книг по ВСЕМУ Ratings.csv
book_ids = ratings["ISBN"].drop_duplicates().tolist()
book_to_col = {isbn: col for col, isbn in enumerate(book_ids)}

print("Rows:", len(user_to_row))
print("Book features:", len(book_to_col))

assert len(book_to_col) == N_BOOKS_EXPECTED


Rows: 61640
Book features: 340556


In [9]:
ratings_model = ratings[
    ratings["User-ID"].isin(user_to_row)
].copy()

row_indices = ratings_model["User-ID"].map(user_to_row).to_numpy(dtype=np.int32)
col_indices = ratings_model["ISBN"].map(book_to_col).to_numpy(dtype=np.int32)

interaction_data = np.ones(len(ratings_model), dtype=np.int8)

X = csr_matrix(
    (interaction_data, (row_indices, col_indices)),
    shape=(len(user_to_row), len(book_to_col)),
    dtype=np.int8,
)

# На случай повторных (user, book) записей CSR мог суммировать значения.
# Для данной задачи нам нужна бинарная матрица.
if X.nnz:
    X.data[:] = 1

age_by_user = users_valid.set_index("User-ID")["Age"]
y = np.array(
    [age_by_user.loc[user_id] for user_id in user_ids],
    dtype=np.float32,
)

print("X.shape =", X.shape)
print("X.nnz =", f"{X.nnz:,}")
print("Плотность =", X.nnz / (X.shape[0] * X.shape[1]))
print("Разреженность =", 1 - X.nnz / (X.shape[0] * X.shape[1]))
print("y.shape =", y.shape)

assert X.shape[1] == N_BOOKS_EXPECTED
assert X.shape[0] == len(y)
assert np.all(X.data == 1)


X.shape = (61640, 340556)
X.nnz = 834,228
Плотность = 3.974052463536763e-05
Разреженность = 0.9999602594753646
y.shape = (61640,)


## 5. Сохраняем подготовленные данные

Это позволит не перестраивать матрицу `X` при каждом повторном запуске ноутбука.


In [10]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

save_npz(PROCESSED_DIR / "user_book_interactions.npz", X)
np.save(PROCESSED_DIR / "age.npy", y)
np.save(PROCESSED_DIR / "user_ids.npy", user_ids)
np.save(PROCESSED_DIR / "book_ids.npy", np.array(book_ids, dtype=object))

print("Сохранено:", PROCESSED_DIR.resolve())


Сохранено: C:\Users\User\Desktop\ML6_Unsupervised_learning_ID_1254805-1\src\data\processed


## 6. Подготовка выборки для ML-экспериментов

Полная sparse matrix сохраняет всех пользователей и все 340 556 признаков. Для наиболее тяжёлых этапов (особенно Random Forest и UMAP) используем воспроизводимую подвыборку пользователей.

Это не меняет структуру признаков: `X` по-прежнему содержит все 340 556 книг.

Если компьютер справляется с экспериментами, `MODEL_SAMPLE_SIZE` можно увеличить или поставить `None`.


In [11]:
if MODEL_SAMPLE_SIZE is None or MODEL_SAMPLE_SIZE >= X.shape[0]:
    model_indices = np.arange(X.shape[0])
else:
    rng = np.random.default_rng(RANDOM_STATE)
    model_indices = np.sort(
        rng.choice(X.shape[0], size=MODEL_SAMPLE_SIZE, replace=False)
    )

X_model = X[model_indices]
y_model = y[model_indices]

print("Размер ML-выборки:", X_model.shape)
print("Количество пользователей:", len(y_model))
print("Количество признаков:", X_model.shape[1])

assert X_model.shape[1] == N_BOOKS_EXPECTED


Размер ML-выборки: (12000, 340556)
Количество пользователей: 12000
Количество признаков: 340556


## 7. Train / Validation / Test

Используем:

- 70% — train;
- 15% — validation;
- 15% — test.

Validation используется для подбора гиперпараметров. Test не участвует в выборе модели.


In [12]:
indices = np.arange(X_model.shape[0])

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

valid_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
)

X_train = X_model[train_idx]
X_valid = X_model[valid_idx]
X_test = X_model[test_idx]

y_train = y_model[train_idx]
y_valid = y_model[valid_idx]
y_test = y_model[test_idx]

print(f"Train: {len(train_idx):,}")
print(f"Valid: {len(valid_idx):,}")
print(f"Test:  {len(test_idx):,}")


Train: 8,400
Valid: 1,800
Test:  1,800


## 8. Метрики регрессии

Возраст — непрерывная целевая переменная, поэтому это задача регрессии.

Используем:

- `R²` — основная метрика, указанная в чек-листе;
- `MAE` — средняя абсолютная ошибка в годах;
- `RMSE` — сильнее штрафует большие ошибки.


In [13]:
def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
    }


# 9. Baseline: средний возраст

Сначала проверим простейшую модель: всегда предсказывать средний возраст обучающей выборки.


In [14]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)

    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
    }

# 10. Ridge на исходных sparse-признаках

Для `Ridge` используем `solver='lsqr'`, который подходит для sparse matrix.

Подбираем `alpha` только на validation.


In [15]:
ridge_alphas = [0.1, 1.0, 10.0, 100.0]
ridge_search = []

for alpha in ridge_alphas:
    model = Ridge(alpha=alpha, solver="lsqr")

    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - start

    start = time.perf_counter()
    valid_pred = model.predict(X_valid)
    predict_time = time.perf_counter() - start

    metrics = regression_metrics(y_valid, valid_pred)

    ridge_search.append({
        "alpha": alpha,
        "R2": metrics["R2"],
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "fit_time_sec": fit_time,
        "predict_time_sec": predict_time,
    })

ridge_search_df = pd.DataFrame(ridge_search).sort_values(
    "R2", ascending=False
).reset_index(drop=True)

display(ridge_search_df)


,alpha,R2,MAE,RMSE,fit_time_sec,predict_time_sec
0,100.0,0.011693,11.421100,14.149757,0.185993,0.000327
1,10.0,-0.006360,11.466139,14.278405,0.450234,0.000361
2,1.0,-0.359272,12.524204,16.594198,1.085972,0.000295
3,0.1,-1.150264,14.374287,20.871272,1.921959,0.000298


In [16]:
best_ridge_alpha = float(ridge_search_df.iloc[0]["alpha"])

ridge = Ridge(alpha=best_ridge_alpha, solver="lsqr")

start = time.perf_counter()
ridge.fit(X_train, y_train)
ridge_fit_time = time.perf_counter() - start

start = time.perf_counter()
ridge_test_pred = ridge.predict(X_test)
ridge_predict_time = time.perf_counter() - start

ridge_metrics = regression_metrics(y_test, ridge_test_pred)

print("Лучший alpha:", best_ridge_alpha)
print(f"R²   = {ridge_metrics['R2']:.6f}")
print(f"MAE  = {ridge_metrics['MAE']:.4f}")
print(f"RMSE = {ridge_metrics['RMSE']:.4f}")
print(f"Fit time = {ridge_fit_time:.3f} sec")
print(f"Predict time = {ridge_predict_time:.3f} sec")


Лучший alpha: 100.0
R²   = 0.017729
MAE  = 11.0831
RMSE = 13.6609
Fit time = 0.181 sec
Predict time = 0.000 sec


# 11. Random Forest на исходных sparse-признаках

Random Forest значительно тяжелее Ridge при сотнях тысяч признаков, поэтому подбор параметров сделан компактным и воспроизводимым.

После получения рабочего baseline при необходимости можно расширить grid.


In [17]:
rf_configs = [
    {"n_estimators": 100, "max_depth": 20, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 20, "min_samples_leaf": 2},
]

rf_search = []

for config in rf_configs:
    rf = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **config,
    )

    start = time.perf_counter()
    rf.fit(X_train, y_train)
    fit_time = time.perf_counter() - start

    start = time.perf_counter()
    valid_pred = rf.predict(X_valid)
    predict_time = time.perf_counter() - start

    metrics = regression_metrics(y_valid, valid_pred)

    rf_search.append({
        **config,
        "R2": metrics["R2"],
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "fit_time_sec": fit_time,
        "predict_time_sec": predict_time,
    })

rf_search_df = pd.DataFrame(rf_search).sort_values(
    "R2", ascending=False
).reset_index(drop=True)

display(rf_search_df)


,n_estimators,max_depth,min_samples_leaf,R2,MAE,RMSE,fit_time_sec,predict_time_sec
0,100,20.0,2,0.004510,11.461882,14.201081,20.581028,0.091003
1,100,20.0,1,0.001340,11.469308,14.223678,23.840913,0.061771
2,100,NaN,1,-0.219556,12.048855,15.718242,706.823672,0.255749


In [18]:
best_rf_params = {
    "n_estimators": int(rf_search_df.iloc[0]["n_estimators"]),
    "max_depth": (
        None
        if pd.isna(rf_search_df.iloc[0]["max_depth"])
        else int(rf_search_df.iloc[0]["max_depth"])
    ),
    "min_samples_leaf": int(rf_search_df.iloc[0]["min_samples_leaf"]),
}

rf = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_rf_params,
)

start = time.perf_counter()
rf.fit(X_train, y_train)
rf_fit_time = time.perf_counter() - start

start = time.perf_counter()
rf_test_pred = rf.predict(X_test)
rf_predict_time = time.perf_counter() - start

rf_metrics = regression_metrics(y_test, rf_test_pred)

print("Лучшие параметры RF:", best_rf_params)
print(f"R²   = {rf_metrics['R2']:.6f}")
print(f"MAE  = {rf_metrics['MAE']:.4f}")
print(f"RMSE = {rf_metrics['RMSE']:.4f}")
print(f"Fit time = {rf_fit_time:.3f} sec")
print(f"Predict time = {rf_predict_time:.3f} sec")


Лучшие параметры RF: {'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 2}
R²   = 0.004893
MAE  = 11.1456
RMSE = 13.7498
Fit time = 20.426 sec
Predict time = 0.050 sec


# 12. Линейное снижение размерности: TruncatedSVD

`TruncatedSVD` работает с исходной CSR sparse matrix напрямую.

Получаем компактное представление:

`340556 признаков → 50 компонентов`

Измеряем отдельно стоимость `fit + transform`.


In [19]:
svd = TruncatedSVD(
    n_components=SVD_COMPONENTS,
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
X_train_svd = svd.fit_transform(X_train)
svd_fit_transform_time = time.perf_counter() - start

start = time.perf_counter()
X_valid_svd = svd.transform(X_valid)
X_test_svd = svd.transform(X_test)
svd_transform_time = time.perf_counter() - start

explained_variance = float(np.sum(svd.explained_variance_ratio_))

print("SVD output shape:", X_train_svd.shape)
print(f"Explained variance ratio sum: {explained_variance:.6f}")
print(f"SVD fit + train transform: {svd_fit_transform_time:.3f} sec")
print(f"SVD valid/test transform: {svd_transform_time:.3f} sec")


SVD output shape: (8400, 50)
Explained variance ratio sum: 0.361711
SVD fit + train transform: 3.868 sec
SVD valid/test transform: 0.085 sec


## 13. Ridge на сжатых признаках

Используем то же train / validation / test разбиение.


In [20]:
ridge_svd = Ridge(alpha=best_ridge_alpha)

start = time.perf_counter()
ridge_svd.fit(X_train_svd, y_train)
ridge_svd_fit_time = time.perf_counter() - start

start = time.perf_counter()
ridge_svd_test_pred = ridge_svd.predict(X_test_svd)
ridge_svd_predict_time = time.perf_counter() - start

ridge_svd_metrics = regression_metrics(y_test, ridge_svd_test_pred)

print(f"R²   = {ridge_svd_metrics['R2']:.6f}")
print(f"MAE  = {ridge_svd_metrics['MAE']:.4f}")
print(f"RMSE = {ridge_svd_metrics['RMSE']:.4f}")
print(f"Model fit time = {ridge_svd_fit_time:.3f} sec")


R²   = 0.000360
MAE  = 11.1894
RMSE = 13.7811
Model fit time = 0.014 sec


## 14. Random Forest на сжатых признаках

In [21]:
rf_svd = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_rf_params,
)

start = time.perf_counter()
rf_svd.fit(X_train_svd, y_train)
rf_svd_fit_time = time.perf_counter() - start

start = time.perf_counter()
rf_svd_test_pred = rf_svd.predict(X_test_svd)
rf_svd_predict_time = time.perf_counter() - start

rf_svd_metrics = regression_metrics(y_test, rf_svd_test_pred)

print(f"R²   = {rf_svd_metrics['R2']:.6f}")
print(f"MAE  = {rf_svd_metrics['MAE']:.4f}")
print(f"RMSE = {rf_svd_metrics['RMSE']:.4f}")
print(f"Model fit time = {rf_svd_fit_time:.3f} sec")


R²   = -0.011157
MAE  = 11.2091
RMSE = 13.8603
Model fit time = 5.113 sec


# 15. UMAP

Для очень высокоразмерной sparse матрицы сначала используем sparse-friendly 50-компонентное представление `TruncatedSVD`, а затем применяем UMAP:

`340556 → 50 → 20`

Это существенно безопаснее по памяти и времени, чем запускать UMAP непосредственно на сотнях тысяч исходных координат.


In [22]:
umap_reducer = umap.UMAP(
    n_components=UMAP_COMPONENTS,
    n_neighbors=UMAP_NEIGHBORS,
    metric="euclidean",
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
X_train_umap = umap_reducer.fit_transform(X_train_svd, y=y_train)
umap_fit_transform_time = time.perf_counter() - start

start = time.perf_counter()
X_valid_umap = umap_reducer.transform(X_valid_svd)
X_test_umap = umap_reducer.transform(X_test_svd)
umap_transform_time = time.perf_counter() - start

print("UMAP train shape:", X_train_umap.shape)
print(f"UMAP fit + train transform: {umap_fit_transform_time:.3f} sec")
print(f"UMAP valid/test transform: {umap_transform_time:.3f} sec")


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP train shape: (8400, 20)
UMAP fit + train transform: 35.321 sec
UMAP valid/test transform: 16.872 sec


## 16. Ridge на UMAP-признаках

In [23]:
ridge_umap = Ridge(alpha=best_ridge_alpha)

start = time.perf_counter()
ridge_umap.fit(X_train_umap, y_train)
ridge_umap_fit_time = time.perf_counter() - start

start = time.perf_counter()
ridge_umap_test_pred = ridge_umap.predict(X_test_umap)
ridge_umap_predict_time = time.perf_counter() - start

ridge_umap_metrics = regression_metrics(y_test, ridge_umap_test_pred)

print(f"R²   = {ridge_umap_metrics['R2']:.6f}")
print(f"MAE  = {ridge_umap_metrics['MAE']:.4f}")
print(f"RMSE = {ridge_umap_metrics['RMSE']:.4f}")
print(f"Model fit time = {ridge_umap_fit_time:.3f} sec")


R²   = 0.009099
MAE  = 11.1555
RMSE = 13.7208
Model fit time = 0.013 sec


## 17. Random Forest на UMAP-признаках

In [24]:
rf_umap = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_rf_params,
)

start = time.perf_counter()
rf_umap.fit(X_train_umap, y_train)
rf_umap_fit_time = time.perf_counter() - start

start = time.perf_counter()
rf_umap_test_pred = rf_umap.predict(X_test_umap)
rf_umap_predict_time = time.perf_counter() - start

rf_umap_metrics = regression_metrics(y_test, rf_umap_test_pred)

print(f"R²   = {rf_umap_metrics['R2']:.6f}")
print(f"MAE  = {rf_umap_metrics['MAE']:.4f}")
print(f"RMSE = {rf_umap_metrics['RMSE']:.4f}")
print(f"Model fit time = {rf_umap_fit_time:.3f} sec")


R²   = -0.013547
MAE  = 11.2428
RMSE = 13.8767
Model fit time = 1.678 sec


# 18. Итоговая таблица сравнения

Главный результат Task 2 — сравнить исходные sparse-признаки со сжатыми представлениями по качеству и времени.

Время reduction (`SVD` / `UMAP`) учитываем отдельно, чтобы сравнение не скрывало стоимость получения embedding.


In [26]:
results = pd.DataFrame([
    {
        "Representation": "Sparse original",
        "Model": "Ridge",
        "Dimensions": X_train.shape[1],
        "R2": ridge_metrics["R2"],
        "MAE": ridge_metrics["MAE"],
        "RMSE": ridge_metrics["RMSE"],
        "Fit time, sec": ridge_fit_time,
        "Reduction time, sec": 0.0,
    },
    {
        "Representation": "Sparse original",
        "Model": "Random Forest",
        "Dimensions": X_train.shape[1],
        "R2": rf_metrics["R2"],
        "MAE": rf_metrics["MAE"],
        "RMSE": rf_metrics["RMSE"],
        "Fit time, sec": rf_fit_time,
        "Reduction time, sec": 0.0,
    },
    {
        "Representation": "SVD (PCA-compatible sparse reduction)",
        "Model": "Ridge",
        "Dimensions": X_train_svd.shape[1],
        "R2": ridge_svd_metrics["R2"],
        "MAE": ridge_svd_metrics["MAE"],
        "RMSE": ridge_svd_metrics["RMSE"],
        "Fit time, sec": ridge_svd_fit_time,
        "Reduction time, sec": svd_fit_transform_time,
    },
    {
        "Representation": "SVD (PCA-compatible sparse reduction)",
        "Model": "Random Forest",
        "Dimensions": X_train_svd.shape[1],
        "R2": rf_svd_metrics["R2"],
        "MAE": rf_svd_metrics["MAE"],
        "RMSE": rf_svd_metrics["RMSE"],
        "Fit time, sec": rf_svd_fit_time,
        "Reduction time, sec": svd_fit_transform_time,
    },
    {
        "Representation": "UMAP",
        "Model": "Ridge",
        "Dimensions": X_train_umap.shape[1],
        "R2": ridge_umap_metrics["R2"],
        "MAE": ridge_umap_metrics["MAE"],
        "RMSE": ridge_umap_metrics["RMSE"],
        "Fit time, sec": ridge_umap_fit_time,
        "Reduction time, sec": svd_fit_transform_time + umap_fit_transform_time,
    },
    {
        "Representation": "UMAP",
        "Model": "Random Forest",
        "Dimensions": X_train_umap.shape[1],
        "R2": rf_umap_metrics["R2"],
        "MAE": rf_umap_metrics["MAE"],
        "RMSE": rf_umap_metrics["RMSE"],
        "Fit time, sec": rf_umap_fit_time,
        "Reduction time, sec": svd_fit_transform_time + umap_fit_transform_time,
    },
])

display(results.round({
    "R2": 6,
    "MAE": 4,
    "RMSE": 4,
    "Fit time, sec": 3,
    "Reduction time, sec": 3,
})
)


,Representation,Model,Dimensions,R2,MAE,RMSE,"Fit time, sec","Reduction time, sec"
0,Sparse original,Ridge,340556,0.017729,11.0831,13.6609,0.181,0.000
1,Sparse original,Random Forest,340556,0.004893,11.1456,13.7498,20.426,0.000
2,SVD (PCA-compatible sparse reduction),Ridge,50,0.000360,11.1894,13.7811,0.014,3.868
3,SVD (PCA-compatible sparse reduction),Random Forest,50,-0.011157,11.2091,13.8603,5.113,3.868
4,UMAP,Ridge,20,0.009099,11.1555,13.7208,0.013,39.189
5,UMAP,Random Forest,20,-0.013547,11.2428,13.8767,1.678,39.189


# 19. Выводы по Task 2

При интерпретации результатов нужно ответить на четыре вопроса:

1. Какая модель лучше предсказывает возраст на исходных sparse-признаках?
2. Что происходит с качеством после снижения размерности?
3. Насколько уменьшается время обучения?
4. Окупает ли выигрыш по времени стоимость вычисления `SVD/UMAP`?

Низкий `R²` сам по себе не является ошибкой: чек-лист проекта показывает пример с очень невысоким качеством и прямо отмечает, что metric value может быть не очень хорошим. Важна корректность эксперимента и сравнение исходного и сжатого представлений.


# 20. Контрольные пункты перед сдачей Task 2

- [x] `Ratings.csv` загружен
- [x] `Users.csv` загружен
- [x] sparse user-book matrix построена
- [x] 340 556 признаков сохранены
- [x] train / validation / test выполнены
- [x] Ridge обучена
- [x] гиперпараметр Ridge подобран по validation
- [x] Random Forest обучен
- [x] гиперпараметры Random Forest подобраны по validation
- [x] линейное снижение размерности выполнено
- [x] UMAP выполнен
- [x] Ridge обучена на сжатых признаках
- [x] Random Forest обучен на сжатых признаках
- [x] качество сравнено
- [x] время обучения и reduction учтено

> В данной реализации для огромной sparse-матрицы вместо плотного классического PCA используется `TruncatedSVD` как sparse-friendly линейный аналог. Это сделано намеренно, чтобы не преобразовывать матрицу 340 556 признаков в плотную форму.
